# Kenya — Audience LLM Exploratory Analysis

**Norman Lear Center × Gates Foundation — Manfluencer Project**

This Kenya notebook mirrors the Nigeria exploratory-analysis handoff for the uploaded audience workbook. It uses the Kenya source workbook directly, normalizes all audience-comment sheets into one table, and produces codebook-style exploratory columns plus summary artifacts.

**Source**
- Audience → `Kenya Audience Analysis Comments.xlsx`

**Outputs**
- `Kenya - Audience LLM Exploratory Data Analyses.xlsx`
- `Kenya Audience LLM Exploratory Report.md`
- `kenya_audience_row_coding.csv`


## 0 — Setup

In [ ]:
from pathlib import Path
import pandas as pd

SOURCE_XLSX = Path('/Users/dhruvbhanderi/Documents/USC/New Research Engineer/Final datasets/Kenya Audience Analysis Comments.xlsx')
OUT_DIR = Path('/Users/dhruvbhanderi/Documents/USC/New Research Engineer/Codex/outputs/kenya_llm_exploratory')
ROW_CODING = OUT_DIR / 'kenya_audience_row_coding.csv'
REPORT = OUT_DIR / 'Kenya Audience LLM Exploratory Report.md'
WORKBOOK = OUT_DIR / 'Kenya - Audience LLM Exploratory Data Analyses.xlsx'

print('source exists:', SOURCE_XLSX.exists(), SOURCE_XLSX)
print('output dir:', OUT_DIR)


## 1 — Run Kenya exploratory analysis

In [ ]:
import subprocess

subprocess.run(['/Users/dhruvbhanderi/.cache/codex-runtimes/codex-primary-runtime/dependencies/python/bin/python3', '/Users/dhruvbhanderi/Documents/USC/New Research Engineer/Codex/outputs/kenya_llm_exploratory/prepare_kenya_llm_data.py'], check=True)
subprocess.run(['/Users/dhruvbhanderi/.cache/codex-runtimes/codex-primary-runtime/dependencies/node/bin/node', '/Users/dhruvbhanderi/Documents/USC/New Research Engineer/Codex/outputs/kenya_llm_exploratory/build_kenya_llm_workbook.mjs'], check=True)

print('wrote:', WORKBOOK)
print('wrote:', REPORT)
print('wrote:', ROW_CODING)


## 2 — Load row-level coded audience data

In [ ]:
audience = pd.read_csv(ROW_CODING, dtype={'comment_id': str})
print(audience.shape)
audience.head()


## 3 — QA checks

In [ ]:
checks = {
    'rows': len(audience),
    'missing_comment_id': int(audience['comment_id'].isna().sum()),
    'missing_comment': int(audience['comment'].isna().sum()),
    'creators': audience['creator'].value_counts().to_dict(),
    'platforms': audience['platform'].value_counts().to_dict(),
}
checks


## 4 — Distribution summaries

In [ ]:
display(pd.crosstab(audience['creator'], audience['sentiment__sentiment']))
display(pd.crosstab(audience['creator'], audience['themes__primary_theme']))
display(pd.crosstab(audience['creator'], audience['stance__audience_response']))


## 5 — Key risk indicators

In [ ]:
risk_summary = audience.groupby('creator').agg(
    comments=('comment_id', 'count'),
    avg_toxicity=('toxicity__toxicity_0_3', 'mean'),
    high_toxicity=('toxicity__toxicity_0_3', lambda s: int((s >= 2).sum())),
    avg_misogyny=('misogyny__intensity_0_3', 'mean'),
    high_misogyny=('misogyny__intensity_0_3', lambda s: int((s >= 2).sum())),
).round(2)
risk_summary


## 6 — Report preview

In [ ]:
print(REPORT.read_text(encoding='utf-8')[:4000])
